In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/rotulado_v1.csv").dropna(subset=["categoria"])

df["descricao"].str.split(" - ").str[0].value_counts().head(15)


descricao
Compra no débito                   201
Transferência enviada pelo Pix      25
Compra Meio De Transporte            9
Aplicação                            8
Pagamento de boleto efetuado         7
Cashback                             7
Transferência recebida pelo Pix      6
Resgate                              5
Pagamento efetuado                   3
Estorno                              2
Pix recebido                         1
Pix enviado                          1
Pix enviado devolvido                1
Crédito liberado                     1
Pagamento de fatura                  1
Name: count, dtype: int64

In [2]:
df["descricao"].str.split().str[-1].value_counts().head(15)

descricao
Bra         201
LTDA         11
Sa            8
PESSOA        6
S.A.          5
01/03         5
-CT           5
A             4
01/02         4
R             3
02/06         3
BERNARDO      3
SBC           3
(0077)        2
(0341)        2
Name: count, dtype: int64

In [3]:
import re

t = re.sub(r"([a-z])(\d)", r"\1 \2", "drogasil2705")

re.sub(r"\d+", "", t)

'drogasil '

In [4]:
re.sub(r"\s+bra$", "", "sao paulo bra")

'sao paulo'

In [5]:
re.sub(r"no estabelecimento\s*+\s*", " ", "no estabelecimento marmel mf indaiatuba")

' marmel mf indaiatuba'

In [6]:
re.sub(r"\d{2}/\d{2}", " ", "01/03")

' '

In [7]:
re.sub(r"\d+ de \d+", " ", "05 de 10")

' '

In [8]:
re.sub(r"-ct$", "", "Basquete-ct")

'Basquete'

In [9]:
re.sub(r"[^\w\s]", " ", "ifd*pizzaria - netflix.com (sp)")

'ifd pizzaria   netflix com  sp '

In [10]:
import sys
sys.path.append("..")            
from src.limpeza import limpar

exemplos = [
    "Compra no débito - Drogasil2705 Sao Bernardo Bra",
    "Compra no débito - No Estabelecimento Marmel Mf Indaiatuba Bra",
    "LATAM AIR*0000 01/03",
    "ALLIANZ SEGU*04 de 10",
    "FORT 585 SP -CT",
    "Pix enviado - PESSOA - NU PAGAMENTOS - IP",
]
for e in exemplos:
    print(f"{e!r:65} -> {limpar(e)!r}")

'Compra no débito - Drogasil2705 Sao Bernardo Bra'                -> 'drogasil sao bernardo'
'Compra no débito - No Estabelecimento Marmel Mf Indaiatuba Bra'  -> 'marmel mf indaiatuba'
'LATAM AIR*0000 01/03'                                            -> 'latam air'
'ALLIANZ SEGU*04 de 10'                                           -> 'allianz segu'
'FORT 585 SP -CT'                                                 -> 'fort sp'
'Pix enviado - PESSOA - NU PAGAMENTOS - IP'                       -> 'pix enviado pessoa nu pagamentos ip'


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X, y = df["descricao"], df["categoria"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # o MESMO cv do baseline

baseline = make_pipeline(
    TfidfVectorizer(lowercase=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
com_limpeza = make_pipeline(
    TfidfVectorizer(preprocessor=limpar),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)

for nome, modelo in [("baseline", baseline), ("com limpeza", com_limpeza)]:
    f1 = cross_val_score(modelo, X, y, cv=cv, scoring="f1_macro")
    acc = cross_val_score(modelo, X, y, cv=cv, scoring="accuracy")
    print(f"{nome:12} F1 macro {f1.mean():.3f} ± {f1.std():.3f} | acurácia {acc.mean():.3f} ± {acc.std():.3f}")

baseline     F1 macro 0.565 ± 0.064 | acurácia 0.610 ± 0.033
com limpeza  F1 macro 0.612 ± 0.066 | acurácia 0.661 ± 0.038


In [ ]:
from sklearn.model_selection import learning_curve

learning_curve(com_limpeza, X, y, cv=cv, scoring="f1_macro", train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0])

(array([106, 213, 319, 426, 533]),
 array([[0.97388593, 0.98778297, 0.98156604, 0.97199265, 0.97213879],
        [0.96068453, 0.96897135, 0.95558927, 0.96027023, 0.9609212 ],
        [0.95910514, 0.95907401, 0.95533146, 0.96226676, 0.94816817],
        [0.9564111 , 0.95909117, 0.95382445, 0.95435875, 0.95070734],
        [0.94595714, 0.94358423, 0.95625067, 0.95559416, 0.95307789]]),
 array([[0.31775304, 0.35390805, 0.29029526, 0.26622023, 0.30053817],
        [0.37827277, 0.39519022, 0.37345585, 0.38014674, 0.32964711],
        [0.54093843, 0.51873185, 0.44802103, 0.4812959 , 0.53498049],
        [0.57948122, 0.58846129, 0.48242452, 0.54510323, 0.5340593 ],
        [0.68317197, 0.68892136, 0.5261    , 0.61173116, 0.55480224]]))